In [ ]:
%load_ext autoreload
%autoreload 2

import logging
from pathlib import Path

import matplotlib.dates as mdates
import numpy as np
import pandas as pd
import scipy.stats as stats
from matplotlib import pyplot as plt

from bollettino_functions import (  # noqa: F401
    collect_points_time_series,
    collect_statistics,
    create_sectors_evolution_mosaic,
    create_time_series_plot,
    load_sectors_results_from_geojson,
    plot_area_panel,
    plot_sector_evolution_boxplots,
    plot_sectors_velocity_area_time_series,
    plot_swimmer_all_sectors,
    plot_velocity_panel,
    plot_velocity_separation_panel,
)
from ppcluster import load_config, setup_logger

%matplotlib widget

logger = setup_logger(logging.INFO, name="ppcx")

input_dir = Path("output_domains/2016_PPCX_Tele")
geojson_dir = input_dir / "kinematic_sectors_geojson"
output_dir = input_dir / "kinematic_sectors_time_series"

config = load_config()

if not geojson_dir.exists():
    raise FileNotFoundError(f"GeoJSON directory not found: {geojson_dir}")

output_dir.mkdir(parents=True, exist_ok=True)

UNIT = "px"
UNIT_SCALE = 1.0  # 1 for px, or the m/px conversion factor

# Temporary dictionary with day to discard per year (e.g. due to bad weather, data issues, etc.)
# TODO: this should be replaced by a more systematic approach (e.g. a metadata file with flags for each date)
DISCARDED_DAYS = {
    "2015": [],
    "2016": [
        # "2016-06-02",
        # "2016-06-14",
        # "2016-07-25",
        # "2016-09-14",
        # "2016-09-15",
        # "2016-09-20",
        # "2016-09-21",
        # "2016-10-01",
        # "2016-10-13",
        # "2016-10-14",
        # "2016-10-15",
        # "2016-10-16",
        # "2016-10-17",
        # "2016-10-18",
        # "2016-10-19",
        # "2016-10-20",
    ],
    "2017": [],
    "2018": [
        "2018-10-04",
        "2018-10-05",
        "2018-10-06",
        "2018-10-07",
    ],
    "2019": [
        "2019-08-08",
        "2019-08-09",
        "2019-08-10",
        "2019-08-11",
        "2019-10-02",
        "2019-10-03",
        "2019-10-04",
        "2019-10-05",
    ],
    "2020": [
        "2020-10-17",
        "2020-10-18",
    ],
    "2021": [],
    "2022": [],
    "2023": [],
    "2024_18mp": [
        "2024-07-24",
    ],
    "2024_24mp": [],
    "2025": [],
}

In [ ]:
# Load results from the final GeoJSON outputs
logger.info(f"Loading sector results from {geojson_dir}...")
results_list = load_sectors_results_from_geojson(geojson_dir)
logger.info(f"Successfully loaded {len(results_list)} results")

In [ ]:
# Collect sector statistics into a DataFrame
df_sectors = collect_statistics(results_list)
logger.info(
    f"Successfully loaded {len(df_sectors)} records from {len(df_sectors['date'].unique())} dates"
)

# Collect point-level time series data
df_points_ts = collect_points_time_series(results_list)
columns_to_drop = ["geometry", "cluster_id", "area"]
df_points_ts = df_points_ts.drop(columns=columns_to_drop, errors="ignore")

logger.info(f"Points time series shape: {df_points_ts.shape}")
logger.info(f"Columns: {df_points_ts.columns.tolist()}")

In [ ]:
_ = plot_sector_evolution_boxplots(
    df_points_ts,
    col="V",
    unit=UNIT,
    sectors=["A", "B", "C", "D"],
    output_path=output_dir / "sectors_velocity_boxplot_evolution.png",
)

In [ ]:
# date_sel = pd.to_datetime("2020-08-02")
# _ = plot_single_date_sector_distributions(
#     df_points_ts,
#     date_sel,
#     sectors=["A", "B", "C"],
#     col="V",
# )


In [ ]:
### SANDBOX for statistical testing of sector separation over time

df_points = df_points_ts.copy()
sec1, sec2 = "A", "B"
rolling_days = None

# Group by date to perform the test daily
dates = sorted(df_points["date"].unique())
p_values = []
valid_dates = []
for date in dates:
    # Extract velocities for the specific day and sectors
    v1 = df_points[(df_points["date"] == date) & (df_points["sector"] == sec1)][
        "V"
    ].dropna()
    v2 = df_points[(df_points["date"] == date) & (df_points["sector"] == sec2)][
        "V"
    ].dropna()

    # Filter out outliers (>+-0.05 percentile)

    # Need minimal data points to run the test
    if len(v1) < 5 or len(v2) < 5:
        continue

    # Mann-Whitney U Test
    # alternative='two-sided' checks if distributions are different
    try:
        stat, p = stats.mannwhitneyu(v1, v2, alternative="two-sided")
        p_values.append(p)
        valid_dates.append(date)
    except ValueError:
        continue

if not valid_dates:
    logger.warning(f"No valid data to test separation for {sec1} vs {sec2}")

# Convert to numpy array for processing
p_values = np.array(p_values)

# Avoid log(0)
p_values = np.where(p_values == 0, 1e-300, p_values)

label = f"{sec1} vs {sec2}"

# Plot -log10(p-value) for better visualization
# High value = High significance (Separated)
# Low value = Low significance (Merged/Same)
significance_score = -np.log10(p_values)

if rolling_days is not None:
    s_sig = pd.Series(significance_score, index=valid_dates)
    significance_score = (
        s_sig.rolling(window=f"{rolling_days}D", center=True).median().values
    )
    label += f" ({rolling_days}d median)"

fig, ax = plt.subplots(figsize=(14, 5))

# Cap the significance score for better visualization
significance_score_plt = np.clip(significance_score, 0, 5)

ax.plot(valid_dates, significance_score_plt, label=label, marker=".", alpha=0.7)

# Threshold lines
# p = 0.05 -> -log10(0.05) ~= 1.3
# p = 0.01 -> -log10(0.01) = 2.0
ax.axhline(
    1.3,
    color="orange",
    linestyle="--",
    alpha=0.5,
    label="p=0.05 (1.3=-log10(0.05)) -> Significant",
)
ax.axhline(
    2.0,
    color="red",
    linestyle="--",
    alpha=0.5,
    label="p=0.01 (2.0=-log10(0.01)) -> Highly Significant",
)

ax.set_ylabel("Significance (-log10 p-value)")
ax.set_title("Robust Statistical Separation (Mann-Whitney U)")
ax.legend(loc="upper left")
ax.grid(True, which="both", linestyle="--", alpha=0.3)

# x-axis formatting
ax.xaxis.set_major_locator(mdates.WeekdayLocator(interval=1))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
ax.xaxis.set_minor_locator(mdates.DayLocator())
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()


In [ ]:
from ppcluster.visualization import get_sector_colors

sector_names = df_sectors["sector"].unique().tolist()
colors = get_sector_colors(
    sector_names,
    colormap="tab10",
)

In [ ]:
# Figure with time series of velocity and area of each sector
output_path = output_dir / "sectors_time_series.png"
create_time_series_plot(
    df_sectors,
    df_points_ts,
    output_path,
    unit=UNIT,
    sector_colors=colors,
    sector_pairs=[("A", "B")],
    rolling_days_area=3,
    rolling_days_sep=3,
    rolling_function="mean",
    show=True,
)


# Manual plot
# fig, axes = plt.subplots(3, 1, figsize=(14, 12))
# ax1, ax2, ax3 = axes
# plot_velocity_panel(
#     ax1,
#     df_sectors,
#     colors,
#     unit=f"{UNIT}/day",
# )
# plot_area_panel(
#     ax2,
#     df_sectors,
#     colors,
#     unit=f"{UNIT}²",
#     rolling_days=3,
#     rolling_function="mean",
# )
# plot_velocity_separation_panel(
#     ax3,
#     df_points=df_points_ts,
#     sector_pairs=[("A", "B")],
#     rolling_days=3,
#     rolling_function="mean",
# )


# plt.tight_layout()
# fig.savefig(output_path, dpi=300)
# fig.savefig(output_path.with_suffix(".svg"), bbox_inches="tight")
# logger.info(f"Saved sector time series to {output_path}")

In [ ]:
output_path = output_dir / "sectors_velocity_area_time_series.png"
fig = plot_sectors_velocity_area_time_series(df_sectors, output_path)

In [ ]:
# Swimmer plot: vertical movement of sectors' centroid over time
fig_separate = plot_swimmer_all_sectors(
    df_sectors,
    colors,
    plot_sector_extent=False,
    subtract_mean=True,
    rolling_days=5,
)
fig_separate.savefig(output_dir / "sectors_centroid_location.png", dpi=300)

In [ ]:
# Generate mosaic for all available data
output_path_mosaic = output_dir / "sectors_evolution_mosaic"
plt.close("all")  # close any existing figures
# create_sectors_evolution_mosaic(
#     df_sectors,
#     output_path_mosaic,
#     image=None,
#     df_points=df_points_ts,
#     max_dates_per_figure=30,
#     ncols=6,
#     nrows=None,
#     velocity_mode="scatter",
#     velocity_cmap="Blues",
#     min_cbar=0.0,
#     max_cbar=10.0,
#     img_kwargs=None,
#     quiver_kwargs=None,
#     scatter_kwargs={"s": 10, "alpha": 0.6},
#     sector_kwargs=None,
#     sector_fill_kwargs={"alpha": 0},
#     sector_edge_kwargs={"linewidth": 5.0},
#     save_svg=False,
#     n_jobs=1,
# )